# V8 · Stage 1.3b — Reliable-θ ablation

**Added on 2026-07-13** in response to Stage 1.4 finding: only 2 of 5 θ
parameters (`D_SEI_solvent`, `k_plating`) are reliably identifiable across
anchors. This notebook tests whether restricting θ to the two reliable
components matches or beats the full 5-dim θ.

**Blocking predecessor**: `01_1c_retrain_clean_split.ipynb` (must complete first
— this ablation uses the same clean grouped dataset).

## Acceptance criterion
Retrain OperatorV7 on `_v8_dataset.parquet` with `theta_norm` restricted to
only the well-identified pair (`D_SEI_solvent`, `k_plating`). Report:

- corpus val + test RMSE
- held-out CALB_0029, EVE_0003, REPT_0028 RMSE
- comparison table across (full-θ, no-θ, reliable-θ, exp baseline)

## Expected outputs
- `outputs/models/pinn_phase3_v8_reliable_theta.pt`
- `outputs/results/reliable_theta_ablation.json`

## Rationale
`k_SEI`, `V_SEI`, `k_LAM_negative` had `well_identified=False` for majority
of anchors. Feeding them into the operator as conditioning inputs may be
adding NOISE rather than SIGNAL. If reliable-θ ≥ full-θ, the abstract
should either:

- describe θ as a 2-dim vector, or
- describe the full 5-dim vector as "physics-informed conditioning
  descriptors" and note that only two components are identifiable.


## 1. Setup

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PROJ = Path("/home/hj/Desktop/PINNs")
sys.path.insert(0, str(PROJ / "Voltaris" / "Data_Exploration"))

V8_DATASET = PROJ / "configs" / "phase3_corpus" / "_v8_dataset.parquet"
V8_FULL_CKPT     = PROJ / "outputs" / "models" / "pinn_phase3_v8_clean.pt"
V8_RELIABLE_CKPT = PROJ / "outputs" / "models" / "pinn_phase3_v8_reliable_theta.pt"

assert V8_DATASET.exists(), "01_1b output missing"
assert V8_FULL_CKPT.exists(), "01_1c full-θ checkpoint missing"


## 2. Method
The v7 dataset encodes θ as `theta_norm` — a 5-element array per row indexed by
alphabetical parameter name (`D_SEI_solvent`, `V_SEI`, `k_LAM_negative`, `k_SEI`,
`k_plating`). To ablate to the reliable pair, we zero out the three unreliable
components in `theta_norm` before training; the model's `theta_dim` stays 5 for
schema compatibility. Effectively:

```
theta_norm_reliable[i] = theta_norm[i] if i ∈ {D_SEI_solvent, k_plating} else 0.0
```

This is the minimal-change ablation. If we wanted to actually shrink the input
dimension, we'd retrain with modified OperatorV7 `theta_dim=2`.

In [ ]:
# Prepare a copy of the v8 dataset with reliable-θ masking
df = pd.read_parquet(V8_DATASET)
THETA_NAMES = ["D_SEI_solvent", "V_SEI", "k_LAM_negative", "k_SEI", "k_plating"]
RELIABLE = {"D_SEI_solvent", "k_plating"}
mask = np.array([1.0 if n in RELIABLE else 0.0 for n in THETA_NAMES],
                dtype=np.float32)
print(f"reliable-θ mask ({THETA_NAMES}): {mask.tolist()}")

df["theta_norm"] = df["theta_norm"].apply(
    lambda arr: (np.asarray(arr, dtype=np.float32) * mask).tolist()
)
V8_RELIABLE_DATASET = PROJ / "configs" / "phase3_corpus" / "_v8_reliable_theta_dataset.parquet"
df.to_parquet(V8_RELIABLE_DATASET, index=False)
print(f"wrote {V8_RELIABLE_DATASET}")


In [ ]:
# Run the trainer on the masked dataset — detach so this notebook is not blocked
import subprocess
LOG = PROJ / "outputs" / "logs" / "phase3_v8_reliable_theta_train.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
print("Launching reliable-θ retrain — detached. See", LOG)
print("Run this from the shell before re-executing this notebook:")
print()
print(f"nohup .venv/bin/python -u Voltaris/Data_Exploration/phase3_v8_train.py \\")
print(f"  --dataset configs/phase3_corpus/_v8_reliable_theta_dataset.parquet \\")
print(f"  --out    outputs/models/pinn_phase3_v8_reliable_theta.pt \\")
print(f"  > {LOG} 2>&1 &")


## 3. Held-out eval + comparison table
(populated after the reliable-θ retrain completes and this notebook is re-run)

In [ ]:
# Load both checkpoints + evaluate held-out cells
if not V8_RELIABLE_CKPT.exists():
    print("Reliable-θ checkpoint not present yet — training in progress.")
else:
    from phase3_v7_validate import load_v7_operator, forecast_v7

    def _eval(ckpt_path):
        m, _ = load_v7_operator(ckpt_path)
        return {f"{mk}_{cid}": float(forecast_v7(m, cid, mk, K=50, forecast_len=500)["rmse_pp_covered"])
                for cid, mk in [("0029","CALB"), ("0003","EVE"), ("0028","REPT")]}

    full_rmse    = _eval(V8_FULL_CKPT)
    reliable_rmse = _eval(V8_RELIABLE_CKPT)

    baseline = pd.read_parquet(PROJ / "outputs/results/baselines_linear_exp.parquet")
    tbl = pd.DataFrame({
        "cell": [b["cell"] for _, b in baseline.iterrows()],
        "exp_baseline_pp":   baseline["rmse_exponential_pp"].tolist(),
        "linear_baseline_pp":baseline["rmse_linear_pp"].tolist(),
        "v8_full_theta_pp":     [full_rmse[c]     for c in baseline["cell"]],
        "v8_reliable_theta_pp": [reliable_rmse[c] for c in baseline["cell"]],
    })
    tbl["reliable_beats_full"] = tbl["v8_reliable_theta_pp"] < tbl["v8_full_theta_pp"]
    tbl.to_parquet(PROJ / "outputs/results/reliable_theta_ablation.parquet", index=False)
    print(tbl.to_markdown(index=False, floatfmt=".2f"))


## 4. Verdict marker

- [ ] **PASS** — reliable-θ matches or beats full-θ (θ can be reduced to 2 dims)
- [ ] **PASS WITH LIMITATIONS** — comparable within noise; ambiguous
- [ ] **FAIL** — full-θ wins on ≥ 2 of 3 cells (all 5 θs contribute despite low identifiability)

**Downstream consequence if PASS**: abstract can honestly say the operator uses
a 2-dim physics-informed conditioning vector, not 5-dim. Simpler + more
defensible.

**Downstream consequence if FAIL**: abstract must acknowledge that some θ
components contribute despite not being individually identifiable — an
open-question caveat worth adding to Limitations.
